# 05 — 1D Suzuki replication with TWA (quantum seeds)

Notebook 01, one for one. Same lattice, potential, ramp, $\tau$, snapshot grid, RNN, window and target.

**The one change:** the Langevin bath is replaced by closed quantum dynamics in the truncated Wigner approximation.

| | Suzuki (nb 01) | This notebook (TWA) |
|---|---|---|
| EOM | $\ddot\phi + \eta\dot\phi - \phi'' + V'(\phi) = \zeta$ | $\ddot\phi - \phi'' + V'(\phi) = 0$ |
| noise | $\langle\zeta\zeta\rangle = 2\eta\theta\,\delta(x-x')\delta(t-t')$, all times | none after $t_i$ |
| initial state | $\phi = \pi = 0$ | Wigner vacuum at $\varepsilon_i$ |

$V(\phi) = (\phi^4 - 2\varepsilon\phi^2)/8$, $\varepsilon = t/\tau$ from $-1.5$ to $1$.

TWA has no bath, so $\eta$ and $\theta$ go together; that is the whole substitution. The seed is the vacuum Gaussian of the quadratic part at $\varepsilon_i$:

$$\langle|\phi_k|^2\rangle = \frac{\hbar}{2\,dx\,\omega_k},\qquad \langle|\pi_k|^2\rangle = \frac{\hbar\,\omega_k}{2\,dx},\qquad \omega_k^2 = \frac{4}{dx^2}\sin^2\!\frac{k\,dx}{2} - \frac{\varepsilon_i}{2}.$$

$\hbar$ is chosen so the $k=0$ seed matches Suzuki's thermal amplitude, $\hbar = 2\theta/\omega_{\min}$. It only sets the log of the amplification time.

**KZ scales change.** Without damping, $\ddot\phi \approx (t/2\tau)\phi$ gives $\hat t = (2\tau)^{1/3}$ and $\hat\xi = \hat t$ ($c = 1$). At $\tau = 128$: $\hat t \approx 6.3$ (TWA) vs $\hat t = \sqrt{2\eta\tau} = 16$ (Langevin).

**Where it can break**
- No damping: the field at $t = \tau$ still oscillates in the wells (radiation, breathers), so the target is noisy.
- The first roll-off from $\phi = 0$ can swing through zero before the deepening wells trap it, so the sign at freeze-out need not be the final sign.
- Walls move ballistically and annihilate instead of freezing.

In [ ]:
import sys, os, time
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
import src.kz as kz

# ---- inputs ----
TAU           = 128
N_SAMPLES     = 3000
DT            = 0.05            # RK4 step (same as classical data)
SNAP_DT       = 1.0             # snapshot spacing (same)
BATCH         = 100
SEED0         = 990304
THETA_CL      = 1e-8            # Suzuki temperature, only used to set hbar
HBAR          = 2*THETA_CL/np.sqrt(1.5/2)   # = 2 theta / omega_min  (~2.3e-8)
OUTDIR        = "../data/1D_twa"
CLASSICAL_DIR = "../data/1D"
FIGDIR        = "../figures/1d_twa"
N_WIN         = 11
FONTSIZE      = 16

# Suzuki parameters, unchanged
kz.DOF, kz.dx      = 1024, 0.5
kz.EPS_I, kz.EPS_F = -1.5, 1.0
kz.TAU             = float(TAU)
# the one change: closed (TWA) dynamics, no bath
kz.eta, kz.theta   = 0.0, 0.0

DOF    = kz.DOF
T_HAT  = (2*TAU)**(1/3)         # underdamped freeze-out time
XI_HAT = T_HAT                  # c = 1
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(FIGDIR, exist_ok=True)
print(f"hbar = {HBAR:.3e}   t_hat(TWA) = {T_HAT:.2f}   t_hat(Langevin) = {np.sqrt(2*TAU):.2f}")

## Generate TWA trajectories
Same loop as `scripts/kz1d/generate_data.py`, with the Wigner vacuum passed as `phi0, pi0` and `eta = theta = 0`. Saved with the same keys as the classical file, so `kz.load` and `kz.window` work unchanged. Skips if the file exists. About 4 GB in memory.

In [ ]:
path = os.path.join(OUTDIR, f"kz_tau{TAU}.npz")
if not os.path.exists(path):
    t0 = time.time()
    n_steps = int(round((kz.EPS_F - kz.EPS_I)*kz.TAU/DT))
    t_hist, dt_actual = kz.make_time_grid(n_steps)
    stride = max(1, int(round(SNAP_DT/dt_actual)))
    ts = t_hist[::stride]

    # Wigner vacuum of the quadratic part at eps_i
    k     = 2*np.pi*np.fft.fftfreq(DOF, d=kz.dx)
    w_k   = np.sqrt((4/kz.dx**2)*np.sin(k*kz.dx/2)**2 - kz.EPS_I/2)
    a_phi = np.sqrt(HBAR/(2*kz.dx*w_k))
    a_pi  = np.sqrt(HBAR*w_k/(2*kz.dx))

    rng = np.random.default_rng(SEED0)
    phi_hist = np.empty((N_SAMPLES, len(ts), DOF), dtype=np.float32)
    for b0 in range(0, N_SAMPLES, BATCH):
        nb = min(BATCH, N_SAMPLES - b0)
        phi0 = np.real(np.fft.ifft(a_phi*np.fft.fft(rng.standard_normal((nb, DOF)), axis=1), axis=1))
        pi0  = np.real(np.fft.ifft(a_pi *np.fft.fft(rng.standard_normal((nb, DOF)), axis=1), axis=1))
        ph, _ = kz.run(t_hist, dt_actual, n_real=nb, stride=stride, phi0=phi0, pi0=pi0)
        phi_hist[b0:b0+nb] = np.transpose(ph, (1, 0, 2)).astype(np.float32)
        el = time.time() - t0
        print(f"  {b0+nb}/{N_SAMPLES}  ({el:.0f}s elapsed, {el/(b0+nb)*(N_SAMPLES-b0-nb):.0f}s left)", flush=True)

    phi_final = phi_hist[:, -1, :]
    n_def = kz.count_defects(phi_final.astype(np.float64))
    np.savez_compressed(path, phi_hist=phi_hist, ts=ts, phi_final=phi_final, n_defects=n_def,
                        TAU=TAU, dt=dt_actual, dx=kz.dx, DOF=DOF, eta=0.0, theta=0.0,
                        hbar=HBAR, seed0=SEED0)
    print(f"-> {path}   defects: mean={n_def.mean():.1f}  sd={n_def.std():.1f}")
    del phi_hist

d    = kz.load(TAU, outdir=OUTDIR)
d_cl = kz.load(TAU, outdir=CLASSICAL_DIR)

## First look: classical vs TWA

In [ ]:
# ---- inputs ----
SAMPLE = 0

fig, ax = plt.subplots(1, 2, figsize=(18, 7))
kz.heatmap(d_cl["phi_hist"][SAMPLE], d_cl["ts"], ax=ax[0])
kz.heatmap(d["phi_hist"][SAMPLE],    d["ts"],    ax=ax[1])
ax[0].set_title("classical (Langevin)", fontsize=FONTSIZE)
ax[1].set_title("quantum (TWA)", fontsize=FONTSIZE)
for a in ax:
    a.set_ylim(-40, TAU)
    a.xaxis.label.set_size(FONTSIZE); a.yaxis.label.set_size(FONTSIZE)
    a.tick_params(labelsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/heatmap_classical_vs_twa.png", dpi=200)
plt.show()

In [ ]:
# ---- inputs ----
T_LIST = [0, 20, 50, 70]
SAMPLE = -1

ts = d["ts"]
x  = kz.grid()
colors = plt.cm.viridis(np.linspace(0, 0.9, len(T_LIST)))

plt.figure(figsize=(12, 5))
for t, c in zip(T_LIST, colors):
    j = int(np.argmin(np.abs(ts - t)))
    sample = d["phi_hist"][SAMPLE, j]
    plt.plot(x, sample/np.abs(sample).max(), color=c, lw=1.2, label=rf"$t = {ts[j]:.0f}$")
plt.plot(x, d["phi_final"][SAMPLE], "k-", lw=2, label=rf"final, $t = \tau = {TAU}$")
plt.axhline(0, color="grey", lw=0.8)
plt.xlim(0, kz.L())
plt.xlabel(r"$x$", fontsize=FONTSIZE)
plt.ylabel(r"$\phi$", fontsize=FONTSIZE)
plt.title("Field growth during the quench (one TWA sample)", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# kink count at t = tau
n_cl  = d_cl["n_defects"]
n_twa = d["n_defects"]
bins  = np.arange(min(n_cl.min(), n_twa.min()), max(n_cl.max(), n_twa.max()) + 2) - 0.5

plt.figure(figsize=(9, 5))
plt.hist(n_cl,  bins=bins, alpha=0.6, label=f"classical  (mean {n_cl.mean():.1f})")
plt.hist(n_twa, bins=bins, alpha=0.6, label=f"TWA  (mean {n_twa.mean():.1f})")
plt.xlabel(r"zero crossings at $t = \tau$", fontsize=FONTSIZE)
plt.ylabel("samples", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.show()

## Model (identical to nb 01)

In [ ]:
def build_model():
    # SimpleRNN over the time window -> final field at every site
    model = keras.Sequential([
        keras.layers.Input(shape=(N_WIN, DOF)),
        keras.layers.SimpleRNN(256, activation="softsign"),
        keras.layers.Dense(DOF),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

KZ_model = build_model()
KZ_model.summary()

In [ ]:
T_SAMPLE = 50

X = kz.window(d, T_SAMPLE).astype("float32")     # (N, 11, 1024)
Y = d["phi_final"].astype("float32")             # (N, 1024)
print(X.shape, Y.shape)
print("input  scale:", np.abs(X).std())
print("target scale:", np.abs(Y).std())

In [ ]:
hist = KZ_model.fit(X, Y, epochs=60, batch_size=10, validation_split=0.2)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hist.history["loss"],     "--", lw=2, label="train")
plt.plot(hist.history["val_loss"], "-",  lw=2, label="validation")
plt.xlabel("epoch", fontsize=FONTSIZE)
plt.ylabel("MSE", fontsize=FONTSIZE)
plt.yscale("log")
plt.title(rf"TWA learning curve, $\tau = {TAU}$, $t = {T_SAMPLE}$", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.show()

In [ ]:
N_VAL  = int(0.2 * len(X))
SAMPLE = 4
MID    = N_WIN // 2
X_val, Y_val = X[-N_VAL:], Y[-N_VAL:]
Y_pred = KZ_model.predict(X_val, verbose=0)

plt.figure(figsize=(12, 4))
plt.plot(kz.grid(), X_val[SAMPLE, MID]/np.abs(X_val[SAMPLE, MID]).max(), "b-", lw=1.0, alpha=0.6,
         label=rf"input, $t = {T_SAMPLE}$ (normalized)")
plt.plot(kz.grid(), Y_val[SAMPLE],  "k-", lw=1.5, label="true final")
plt.plot(kz.grid(), Y_pred[SAMPLE], "r-", lw=1.5, label="predicted final")
plt.axhline(0, color="grey", lw=0.8)
plt.xlim(0, kz.L())
plt.xlabel(r"$x$", fontsize=FONTSIZE)
plt.ylabel(r"$\phi$", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/Predicted_Final_1d_twa.png", dpi=300)
plt.show()

In [ ]:
x_true = kz.defect_positions(Y_val[SAMPLE].astype(np.float64))
x_pred = kz.defect_positions(Y_pred[SAMPLE].astype(np.float64))

plt.figure(figsize=(12, 2.5))
plt.plot(x_true, np.zeros(len(x_true)), "o", mfc="none", mec="k", ms=12, mew=2, label=f"true ({len(x_true)} kinks)")
plt.plot(x_pred, np.zeros(len(x_pred)), "o", color="r", ms=6, label=f"predicted ({len(x_pred)} kinks)")
plt.xlim(0, kz.L())
plt.yticks([])
plt.xlabel(r"$x$", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2, loc="upper center", bbox_to_anchor=(0.5, 1.6), ncol=2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/defect_location_prediction_twa.png", dpi=300)
plt.show()

## Learning curves at several input times (same as nb 01, unnormalized input)

In [ ]:
# ---- inputs ----
T_LIST = [-80, -5, 20, 50, 70]
COLORS = ["red", "darkorange", "purple", "blue", "green"]
EPOCHS = 60

Y_all = d["phi_final"].astype("float32")
hists = {}
for t in T_LIST:
    Xt = kz.window(d, t).astype("float32")
    m  = build_model()
    h  = m.fit(Xt, Y_all, epochs=EPOCHS, batch_size=10, validation_split=0.2, verbose=0)
    hists[t] = h.history
    print(f"t = {t:4d}   final train = {h.history['loss'][-1]:.3f}   final val = {h.history['val_loss'][-1]:.3f}")

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
plt.figure(figsize=(8, 8))
for t, c in zip(T_LIST, COLORS):
    plt.plot(epochs, hists[t]["loss"],     "--", color=c, lw=1.8)
    plt.plot(epochs, hists[t]["val_loss"], "-",  color=c, lw=1.8, label=rf"$t = {t}$")
plt.plot([], [], "k--", label="training")
plt.plot([], [], "k-",  label="validation")
plt.xlabel("Epoch", fontsize=FONTSIZE)
plt.ylabel("Training / validation loss (MSE)", fontsize=FONTSIZE)
plt.title(rf"TWA, $\tau_Q = {TAU}$", fontsize=FONTSIZE + 2)
plt.xlim(epochs.min(), epochs.max())
plt.ylim(0, None)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 3, loc="upper right")
plt.tight_layout()
plt.savefig(f"{FIGDIR}/final_t_validation_curves_twa.png", dpi=300)
plt.show()

## Low-pass test
Same as nb 01, but $\hat\xi$ is the underdamped one, $\hat\xi = (2\tau)^{1/3}$.

In [ ]:
# ---- inputs ----
T_LIST  = [-10, 0, 10, 20, 35, 50]
KC_MULT = np.geomspace(0.1, 4, 12)      # cutoffs in units of 1/xi_hat
EPOCHS  = 60

k_hat   = 1 / XI_HAT
Y       = d["phi_final"].astype("float32")
n_train = int(0.8 * len(Y))

val_full, val_cut = {}, {}
for t in T_LIST:
    X = kz.window(d, t).astype("float32")
    k = 2 * np.pi * np.fft.rfftfreq(X.shape[-1], d=kz.dx)

    Xn = X / X[:n_train].std()
    m  = build_model()
    h  = m.fit(Xn, Y, epochs=EPOCHS, batch_size=10, validation_split=0.2, verbose=0)
    val_full[t] = h.history["val_loss"][-1]
    print(f"t = {t:3d} ({t / T_HAT:.1f} t_hat)   no filter: val = {val_full[t]:.4f}")

    F = np.fft.rfft(X, axis=-1)
    val_cut[t] = []
    for mult in KC_MULT:
        Ff = F.copy()
        Ff[..., k > mult * k_hat] = 0
        Xf = np.fft.irfft(Ff, n=X.shape[-1], axis=-1).astype("float32")
        Xf = Xf / Xf[:n_train].std()
        m  = build_model()
        h  = m.fit(Xf, Y, epochs=EPOCHS, batch_size=10, validation_split=0.2, verbose=0)
        val_cut[t].append(h.history["val_loss"][-1])
        print(f"      kc = {mult:5.2f}/xi_hat:  val = {val_cut[t][-1]:.4f}")

colors = plt.cm.plasma(np.linspace(0, 0.85, len(T_LIST)))
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
for t, c in zip(T_LIST, colors):
    lab = rf"$t = {t / T_HAT:.1f}\,\hat t$"
    ax[0].plot(KC_MULT, val_cut[t], "o-", color=c, ms=8, lw=2, label=lab)
    ax[0].axhline(val_full[t], color=c, ls="--", lw=1.2)
    ax[1].plot(KC_MULT, np.array(val_cut[t]) / val_full[t], "o-", color=c, ms=8, lw=2, label=lab)
ax[0].set_ylabel("validation MSE  (dashed: no filter)", fontsize=FONTSIZE)
ax[1].set_ylabel("validation MSE / no-filter MSE", fontsize=FONTSIZE)
ax[1].axhline(1, color="k", ls="--", lw=1.5)
for a in ax:
    a.axvline(1, color="grey", ls=":", lw=1.5)
    a.set_xscale("log")
    a.set_xlim(KC_MULT.min(), KC_MULT.max())
    a.set_xlabel(r"cutoff $k_c\,\hat\xi$", fontsize=FONTSIZE)
    a.tick_params(labelsize=FONTSIZE - 2)
    a.legend(fontsize=FONTSIZE - 3)
fig.suptitle(rf"TWA, low-pass filtered input, $\tau_Q = {TAU}$", fontsize=FONTSIZE + 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/lowpass_training_twa.png", dpi=300)
plt.show()

## Headline: classical vs TWA, same model, same inputs
11-snapshot window, normalized input (as in nb 01's last test). Dashed = persistence baseline: the sign of the input at time $t$ used as the prediction.

In [ ]:
# ---- inputs ----
T_LIST = [-40, -20, -10, -5, 0, 5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70]
EPOCHS = 60
DATA   = {"classical (Langevin)": d_cl, "quantum (TWA)": d}
MID    = N_WIN // 2

val_loss = {lab: [] for lab in DATA}
sign_err = {lab: [] for lab in DATA}
base_err = {lab: [] for lab in DATA}
for lab, dd in DATA.items():
    Y     = dd["phi_final"].astype("float32")
    N_VAL = int(0.2 * len(Y))
    for t in T_LIST:
        X = kz.window(dd, t).astype("float32")
        X = X / X[:-N_VAL].std()
        m = build_model()
        h = m.fit(X, Y, epochs=EPOCHS, batch_size=10, validation_split=0.2, verbose=0)
        P = m.predict(X[-N_VAL:], verbose=0)
        val_loss[lab].append(h.history["val_loss"][-1])
        sign_err[lab].append(((P > 0) != (Y[-N_VAL:] > 0)).mean())
        base_err[lab].append(((X[-N_VAL:, MID] > 0) != (Y[-N_VAL:] > 0)).mean())
        print(f"{lab:>22s}  t = {t:4d}   val MSE = {val_loss[lab][-1]:.4f}   "
              f"sign error = {sign_err[lab][-1]:.4f}   baseline = {base_err[lab][-1]:.4f}")

In [ ]:
T = np.array(T_LIST)
fig, ax = plt.subplots(1, 2, figsize=(18, 6.5))
for lab, c in zip(DATA, ["C0", "C3"]):
    ax[0].plot(T, val_loss[lab], "o-", color=c, ms=9, lw=2, label=lab)
    ax[1].plot(T, sign_err[lab], "o-", color=c, ms=9, lw=2, label=lab + "  (RNN)")
    ax[1].plot(T, base_err[lab], "--", color=c, lw=1.8, label=lab + "  (sign of input)")

# ---- reference lines (edit freely) ----
for a in ax:
    a.axvline(16.0, color="C0", ls=":", lw=1.5)
    a.axvline(6.35, color="C3", ls=":", lw=1.5)
ax[1].text(16.5, 0.45, r"$\hat t_{\rm Langevin}=16$", color="C0", fontsize=FONTSIZE - 2)
ax[1].text(6.8,  0.40, r"$\hat t_{\rm TWA}=6.3$",     color="C3", fontsize=FONTSIZE - 2)

ax[0].set_ylabel("validation MSE", fontsize=FONTSIZE)
ax[1].set_ylabel("per-site sign error", fontsize=FONTSIZE)
ax[1].set_ylim(0, 0.55)
for a in ax:
    a.set_xlim(T.min(), T.max())
    a.set_xlabel(r"input time $t$", fontsize=FONTSIZE)
    a.tick_params(labelsize=FONTSIZE - 2)
    a.legend(fontsize=FONTSIZE - 3)
fig.suptitle(rf"$\tau_Q = {TAU}$: Langevin vs TWA", fontsize=FONTSIZE + 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/classical_vs_twa_sign_error.png", dpi=300)
plt.show()